# MCP 101 with Harborlight Insurance Agency

## Goal

Use the real `harborlight_mcp` package to inspect synthetic renewals, calculate a proposed premium change, and then prove the same capability works through an MCP client/server exchange.

**Harborlight Insurance Agency is fictional. Every record in this tutorial is synthetic. This is an educational protocol example, not insurance advice or production software.**

## Setup

Run this notebook from the repository root after installing the project:

```text
python -m pip install -e ".[dev]"
```

The notebook imports the installed package. It does not redefine the service or MCP tool implementation.

In [ ]:
import json
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

from harborlight_mcp import calculate_premium_change, list_upcoming_renewals

repository_root = Path.cwd()
if not (repository_root / "pyproject.toml").is_file():
    repository_root = repository_root.parent
assert (repository_root / "pyproject.toml").is_file()

print(f"Python: {sys.version.split()[0]}")
print(f"Official MCP Python SDK: {version('mcp')}")
print(f"Tutorial package: {version('harborlight-mcp')}")

## Steps

### 1. Ask the service for upcoming renewals

The packaged dataset has a fixed snapshot date of `2026-07-01`. A 30-day inclusive window is deterministic and returns three fictional records.

In [ ]:
renewal_result = list_upcoming_renewals(days=30)
print(json.dumps(renewal_result, indent=2))

### 2. Calculate a proposed premium change

Money is passed as whole cents. The service returns a JSON-compatible structure and rounds the percentage to two decimal places.

In [ ]:
premium_result = calculate_premium_change(
    current_cents=120_000,
    renewal_cents=126_000,
)
print(json.dumps(premium_result, indent=2))

### 3. Cross the MCP protocol boundary

The calls above isolate and explain the business logic; by themselves they are **not** MCP protocol tests. The next cell runs the repository's example client. That client launches a separate stdio server, initializes an MCP `ClientSession`, lists tools, and calls `calculate_premium_change` through MCP.

In [ ]:
client_run = subprocess.run(
    [sys.executable, str(repository_root / "examples" / "protocol_client.py")],
    check=True,
    capture_output=True,
    text=True,
    timeout=60,
    cwd=repository_root,
)
print(client_run.stdout)

## Checks

In [ ]:
assert renewal_result["fictional"] is True
assert [item["policy_id"] for item in renewal_result["renewals"]] == [
    "FIC-HLA-1001",
    "FIC-HLA-1002",
    "FIC-HLA-1003",
]
assert premium_result == {
    "current_cents": 120_000,
    "renewal_cents": 126_000,
    "change_cents": 6_000,
    "change_percent": 5.0,
    "direction": "increase",
}
assert "Initialized MCP session." in client_run.stdout
assert "calculate_premium_change, list_upcoming_renewals" in client_run.stdout
print("All notebook checks passed.")

## Next steps

1. Read `src/harborlight_mcp/services.py` to see the deterministic data and calculation rules.
2. Read `src/harborlight_mcp/server.py` to see the small MCP registration layer.
3. Read `tests/test_protocol.py` to see tool discovery and two invocations tested across stdio.
4. Try the same two tools in MCP Inspector using the README instructions.

Keep extensions within the tutorial boundary: synthetic data, read-only behavior, and no insurance decisions or advice.